In [0]:
%python
from pyspark.sql.functions import to_date, sum as spark_sum

df_gold = (
    spark.table("workspace.lab_db.silver_orders")
    .withColumn("order_date", to_date("order_timestamp"))
    .groupBy("customer_id", "order_date")
    .agg(
        spark_sum("amount").alias("daily_revenue")
    )
)

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.lab_db.gold_daily_revenue")

In [0]:
%python
spark.sql("""
SELECT *
FROM workspace.lab_db.gold_daily_revenue
ORDER BY customer_id, order_date
""").show(truncate=False)

In [0]:
%python
spark.sql("""
DESCRIBE workspace.lab_db.gold_daily_revenue
""").show(truncate=False)

In [0]:
CREATE OR REPLACE TABLE workspace.lab_db.gold_dim_customers (
    customer_id INT,
    name STRING,
    email STRING,
    effective_date DATE,
    end_date DATE,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
INSERT INTO workspace.lab_db.gold_dim_customers
VALUES
(101, 'Alice Smith', 'alice@example.com', current_date(), NULL, true),
(102, 'Bob Jones', 'bob@example.com', current_date(), NULL, true);

In [0]:
CREATE OR REPLACE TEMP VIEW staging_updates AS
SELECT
    102 AS customer_id,
    'Bob Jones' AS name,
    'bob_new@example.com' AS email
UNION ALL
SELECT
    103 AS customer_id,
    'Charlie Brown' AS name,
    'charlie@example.com' AS email;

In [0]:
MERGE INTO workspace.lab_db.gold_dim_customers AS target
USING (
    SELECT
        customer_id AS merge_key,
        customer_id,
        name,
        email,
        current_date() AS effective_date
    FROM staging_updates

    UNION ALL

    SELECT
        NULL AS merge_key,
        customer_id,
        name,
        email,
        current_date() AS effective_date
    FROM staging_updates
) AS source

ON target.customer_id = source.merge_key

WHEN MATCHED AND target.is_current = true THEN
    UPDATE SET
        target.is_current = false,
        target.end_date = current_date()

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        name,
        email,
        effective_date,
        end_date,
        is_current
    )
    VALUES (
        source.customer_id,
        source.name,
        source.email,
        source.effective_date,
        NULL,
        true
    );

In [0]:
SELECT
    customer_id,
    name,
    email,
    effective_date,
    end_date,
    is_current
FROM workspace.lab_db.gold_dim_customers
ORDER BY customer_id, effective_date;